In [ ]:
import tmdbsimple as tmdb
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
tmdb.API_KEY = os.getenv('TMDB_API_KEY', 'your_api_key_here')

In [ ]:
import requests
import time
import re
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
API_KEY = os.getenv('TMDB_API_KEY', 'your_api_key_here')
url = "https://api.themoviedb.org/3/search/movie"

def clean_title(title):
    # Remove year in parentheses, e.g. "Jumanji (1995)" -> "Jumanji"
    return re.sub(r"\s*\(\d{4}\)", "", title).strip()

def get_tmdb_data(title, max_retries=5):
    clean_query = clean_title(title)
    params = {
        "api_key": API_KEY,
        "query": clean_query
    }
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json"
    }
    delay = 1  # start delay 1 sec

    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=10)
            response.raise_for_status()
            data = response.json()
            if data['results']:
                return data['results'][0]
            else:
                print(f"No results for: {title}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"[Attempt {attempt+1}] Error fetching '{title}': {e}. Retrying in {delay}s...")
            time.sleep(delay)
            delay *= 2

    print(f"Failed to fetch data for '{title}' after {max_retries} attempts.")
    return None

# Test on a single movie
metadata = get_tmdb_data("Jumanji (1995)")
if metadata:
    print(f"Title: {metadata['title']}, Release Date: {metadata.get('release_date', 'N/A')}")
else:
    print("No metadata found.")

Title: Jumanji, Release Date: 1995-12-15


In [ ]:
import pandas as pd
import time
import re
import requests
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
API_KEY = os.getenv('TMDB_API_KEY', 'your_api_key_here')
SEARCH_URL = "https://api.themoviedb.org/3/search/movie"

def clean_title(title):
    return re.sub(r"\s*\(\d{4}\)", "", title).strip()

def get_tmdb_data(title, max_retries=5):
    query = clean_title(title)
    params = {
        "api_key": API_KEY,
        "query": query
    }
    headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json"
    }
    delay = 1
    for attempt in range(max_retries):
        try:
            response = requests.get(SEARCH_URL, params=params, headers=headers, timeout=10)
            response.raise_for_status()
            data = response.json()
            if data['results']:
                return data['results'][0]
            else:
                print(f"No results for: {title}")
                return None
        except requests.exceptions.RequestException as e:
            print(f"[Attempt {attempt+1}] Error fetching '{title}': {e}. Retrying in {delay}s...")
            time.sleep(delay)
            delay *= 2
    print(f"Failed to fetch data for '{title}' after {max_retries} attempts.")
    return None

In [9]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load the dataset
movies = pd.read_csv('movies.csv')  # Assuming columns: movieId, title, genres
print(movies.head())


   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [10]:
enriched_data = []
for i, row in movies.iterrows():
    title = row['title']
    print(f"Fetching metadata for: {title} ({i+1}/{len(movies)})")
    metadata = get_tmdb_data(title)
    if metadata:
        enriched_data.append({**row.to_dict(), **metadata})
    else:
        enriched_data.append(row.to_dict())  # keep original if no metadata

    if (i + 1) % 50 == 0:  # Save progress every 50 movies
        pd.DataFrame(enriched_data).to_csv("movies_with_metadata_partial.csv", index=False)
        print(f"Saved progress at {i+1} movies.")

    time.sleep(1)  # be gentle on the API

Fetching metadata for: Toy Story (1995) (1/62423)
Fetching metadata for: Jumanji (1995) (2/62423)
Fetching metadata for: Grumpier Old Men (1995) (3/62423)
Fetching metadata for: Waiting to Exhale (1995) (4/62423)


KeyboardInterrupt: 

In [13]:
enriched_df = pd.read_csv('movies_with_metadata.csv')


In [16]:
# Combine relevant textual data into a single 'soup' for TF-IDF processing
enriched_df["soup"] = (
    enriched_df["genres"].fillna("") + " " +
    enriched_df["overview"].fillna("") + " " +
    enriched_df["original_language"].fillna("") + " " +
    enriched_df["original_title"].fillna("") + " " +
    enriched_df["vote_average"].fillna(0).astype(str)
)

# Apply TF-IDF vectorization to the soup
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(enriched_df["soup"])

# Output the shape of the TF-IDF matrix
print(tfidf_matrix.shape)  # (num_movies, num_unique_words_in_soup)

(20800, 51555)


In [17]:
from sklearn.neighbors import NearestNeighbors

# Fit on TF-IDF vectors
knn = NearestNeighbors(n_neighbors=10, metric='cosine', algorithm='brute')
knn.fit(tfidf_matrix)

# Get recommendations for movie index 0
distances, indices = knn.kneighbors(tfidf_matrix[0])
print(movies['title'].iloc[indices[0]])

0                         Toy Story (1995)
14813                   Toy Story 3 (2010)
3021                    Toy Story 2 (1999)
10169       40-Year-Old Virgin, The (2005)
7758          Love Finds Andy Hardy (1938)
20072      Andy Hardy's Double Life (1942)
19937    Woody Allen: A Documentary (2012)
9680                Amongst Friends (1993)
1076          Rebel Without a Cause (1955)
16236                       Win Win (2011)
Name: title, dtype: object


In [18]:
from difflib import get_close_matches

def get_closest_title(input_title, title_list):
    matches = get_close_matches(input_title, title_list, n=1, cutoff=0.8)
    return matches[0] if matches else None

In [19]:
def recommender(input_text, knn_model=knn, tfidf_vectorizer=tfidf, movies_df=movies):
    input_text = input_text.strip().lower()
    title_list = movies_df["title"].str.lower().tolist()
    matched_title = get_closest_title(input_text, title_list)

    if matched_title:
        idx = movies_df[movies_df["title"].str.lower() == matched_title].index[0]
        query_vector = tfidf_matrix[idx]
    else:
        query_vector = tfidf_vectorizer.transform([input_text])
    
    distances, indices = knn_model.kneighbors(query_vector)
    results = movies_df.iloc[indices[0]].copy()
    results["similarity"] = 1 - distances[0]
    return results[["title", "genres", "similarity"]]

In [37]:
print(recommender("Up"))

                                                   title           genres  \
20792                                      Always (1985)     Comedy|Drama   
20793  Trinity: Gambling for High Stakes (Odds and Ev...     Comedy|Crime   
20794                                  Free Style (2008)            Drama   
20795                     $ellebrity (Sellebrity) (2012)      Documentary   
20796                           Macabre (Macabro) (1980)  Horror|Thriller   
20797                    Punk's Dead: SLC Punk! 2 (2014)           Comedy   
20798                            Chinese Hercules (1973)     Action|Drama   
20799                                           Q (2011)            Drama   
20784     Demons 2 (Dèmoni 2... l'incubo ritorna) (1986)           Horror   
20785                                   The Phynx (1970)           Comedy   

       similarity  
20792         0.0  
20793         0.0  
20794         0.0  
20795         0.0  
20796         0.0  
20797         0.0  
20798       